# Assignment 3: Hidden Markov Models for POS Tagging and Text Generation
## CNG463 - Introduction to Natural Language Processing
### METU NCC Computer Engineering | Fall 2025-26

**Student Name:**  
**Student ID:**  
**Due Date:** 14 December 2025 (Sunday) before midnight

---

## Overview

This assignment focuses on:
1. Building **supervised**, **unsupervised**, and **semi-supervised** HMM models for Part-of-Speech (POS) tagging
2. Implementing **5-fold cross-validation** to evaluate model performance
3. Comparing the three training approaches using per-tag and overall accuracy
4. Using HMMs for **creative text generation** with temperature-based sampling

**Note:** You will use the Brown corpus with universal POS tagset. Start with 5000 sentences for debugging, then run on the full corpus (or as much as Colab can handle).

**Grading:**
- Helper Functions: **10 pts**
  - `remove_tags()`: 2 pts
  - `evaluate_tagger()`: 5 pts
  - Results display: 3 pts
- Semi-supervised HMM: **20 pts**
- 5-fold cross-validation: **25 pts**
  - Supervised HMM: 10 pts
  - Unsupervised HMM (Baum-Welch): 10 pts
  - Semi-supervised HMM
  - Evaluate and Accumulate Results: 5 pts
- Report on Results: **5 pts**
- Text Generation: **24 pts**
  - `sample_state()`: 7 pts
  - `sample_word()`: 7 pts
  - `generate_text_from_word()`: 10 pts
- Written Questions (4 × 4 pts): **16 pts**
- **Total: 100 pts**

---

## Pre-Submission Checklist

- [ ] Name and student ID at top
- [ ] No cells are added or removed
- [ ] All TODO sections completed
- [ ] All questions answered
- [ ] Code runs without errors
- [ ] Results tables included
- [ ] Run All before saving

## Setup and Imports

In [28]:
# Standard libraries
import numpy as np
import random
from collections import defaultdict

# NLTK for corpus and HMM
import nltk
from nltk.tag import hmm

# Scikit-learn for cross-validation
from sklearn.model_selection import KFold

# Set random seed for reproducibility
seed = 42
np.random.seed(seed)
random.seed(seed)

---

# Task 1: HMM-based POS Tagging (72 points)

In this part, you will implement three different approaches to training HMM models for POS tagging:
1. **Supervised learning**: Uses fully labelled data
2. **Unsupervised learning**: Uses unlabelled data with Baum-Welch algorithm
3. **Semi-supervised learning**: Combines a small amount of labelled data with larger unlabelled data

## 1.1: Load the Brown Corpus

Load the Brown corpus with universal POS tagset. Start with 5000 sentences for testing, then increase to the maximum your environment can handle.

In [29]:
# Download required NLTK data
from nltk.corpus import brown
nltk.download('brown')
nltk.download('universal_tagset')

NUM_SENTENCES = 5000  # Start with 5000 sentences and increase later

all_data = list(brown.tagged_sents(tagset="universal"))[:5000]

print(f"Total sentences loaded: {len(all_data)}")
print(f"\nExample sentence:")
print(all_data[0])

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


Total sentences loaded: 5000

Example sentence:
[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


## 1.2: Remove Tags (2 points)

Implement the `remove_tags()` function that converts tagged sentences to untagged format required by NLTK's unsupervised training.

**Input:** List of tagged sentences `[[(word, tag), ...], ...]`  
**Output:** List of untagged sentences `[[(word, None), ...], ...]`

In [30]:
def remove_tags(tagged_sents):
    """
    Remove tags from tagged sentences to create untagged data.

    Args:
        tagged_sents: List of tagged sentences [[(word, tag), ...], ...]

    Returns:
        List of untagged sentences [[(word, None), ...], ...] (required format for NLTK)
    """
    untagged_sents = []
    for sent in tagged_sents:
        new_sent = []
        for word, tag in sent:
            new_sent.append((word, None))
        untagged_sents.append(new_sent)
    return untagged_sents


## 1.3: Evaluate Tagger (5 points)

Implement the `evaluate_tagger()` function that evaluates a trained tagger on test data and returns per-tag accuracy statistics.

**Input:**
- `tagger`: Trained HMM tagger
- `test_data`: List of tagged test sentences

**Output:** Dictionary with per-tag statistics `{tag: {'correct': count, 'total': count}}`

In [45]:
def evaluate_tagger(tagger, test_data):
    """
    Evaluate tagger and return per-tag accuracy.

    Args:
        tagger: Trained HMM tagger
        test_data: List of tagged sentences for testing

    Returns:
        dict of {tag: {'correct': count, 'total': count}}
    """
    tag_stats = defaultdict(lambda: {'correct': 0, 'total': 0})

    for sent in test_data:
        words = [word for word, tag in sent]
        true_tags = [tag for word, tag in sent]

        try:
            predicted = tagger.tag(words)
            predicted_tags = [tag for word, tag in predicted]

            for true_tag, pred_tag in zip(true_tags, predicted_tags):
                tag_stats[true_tag]['total'] += 1
                if true_tag == pred_tag:
                    tag_stats[true_tag]['correct'] += 1

        except:
            for true_tag in true_tags:
                tag_stats[true_tag]['total'] += 1

    return tag_stats

## 1.4: Semi-supervised HMM Training (20 points)

Implement semi-supervised HMM training that combines a small amount of labelled data (1%) with larger unlabelled data (99%).

**Steps:**
1. Split data: 1% tagged, 99% untagged
2. Train supervised model on 1% tagged data
3. Use this model to initialise Baum-Welch on 99% untagged data
4. Refine with unsupervised learning

In [46]:
def train_semi_supervised(tagged_data, percent_tagged=1.0):
    """
    Train HMM with a mix of tagged and untagged data.

    Args:
        tagged_data: List of tagged sentences
        percent_tagged: Percentage of data to keep tags (default 1.0%)

    Returns:
        Trained HMM tagger
    """
    split_idx = int(len(tagged_data) * percent_tagged / 100)
    if split_idx == 0:
        split_idx = 1

    tagged_part = tagged_data[:split_idx]
    untagged_part = remove_tags(tagged_data[split_idx:])

    if len(tagged_part) == 0 or len(untagged_part) == 0:
        raise ValueError("Invalid split for semi-supervised training")

    tags = []
    words = []

    for sent in tagged_data:
        for w, t in sent:
            if t not in tags:
                tags.append(t)
            if w not in words:
                words.append(w)

    trainer = hmm.HiddenMarkovModelTrainer(
        states=tags,
        symbols=words
    )

    model = trainer.train_supervised(tagged_part)

    model = trainer.train_unsupervised(
        untagged_part,
        model=model,
        max_iterations=5
    )

    return model


## 1.5: 5-Fold Cross-Validation (25 Points)

Implement 5-fold cross-validation to train and evaluate all three models. This ensures robust performance estimates.

**Steps:**
1. Split data into 5 folds
2. For each fold:
   - Train
    - supervised (`trainer.train_supervised()`)
    - unsupervised (`trainer.train_unsupervised()`)
    - semi-supervised models (`train_semi_supervised()`)
   - Evaluate on test fold
   - Accumulate results
3. Calculate average accuracy across all folds

In [47]:
# Prepare for 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=seed)

# Storage for results across folds
results = {
    'unsupervised': defaultdict(lambda: {'correct': 0, 'total': 0}),
    'supervised': defaultdict(lambda: {'correct': 0, 'total': 0}),
    'semi_supervised': defaultdict(lambda: {'correct': 0, 'total': 0})
}
fold_num = 1

for train_idx, test_idx in kf.split(all_data):
    print("\n" + "-" * 40)
    print("Running fold", fold_num)
    print("-" * 40)

    train_data = [all_data[i] for i in train_idx]
    test_data = [all_data[i] for i in test_idx]

    print("Train size:", len(train_data))
    print("Test size:", len(test_data))

    all_tags = []
    all_symbols = []

    for sent in train_data:
        for word, tag in sent:
            if tag not in all_tags:
                all_tags.append(tag)
            if word not in all_symbols:
                all_symbols.append(word)

    print("\nUnsupervised model is training")
    try:
        untagged_train = remove_tags(train_data)
        trainer = hmm.HiddenMarkovModelTrainer(
            states=all_tags,
            symbols=all_symbols
        )

        unsup_tagger = trainer.train_unsupervised(untagged_train,max_iterations=5)

        stats_unsup = evaluate_tagger(unsup_tagger, test_data)

        for tag in stats_unsup:
            results['unsupervised'][tag]['correct'] += stats_unsup[tag]['correct']
            results['unsupervised'][tag]['total'] += stats_unsup[tag]['total']

        print("Unsupervised model done")
    except Exception as e:
        print("Unsupervised failed:", e)

    print("\nSupervised model is training")
    try:
        trainer = hmm.HiddenMarkovModelTrainer(
            states=all_tags,
            symbols=all_symbols
        )

        sup_tagger = trainer.train_supervised(train_data)
        stats_sup = evaluate_tagger(sup_tagger, test_data)

        for tag in stats_sup:
            results['supervised'][tag]['correct'] += stats_sup[tag]['correct']
            results['supervised'][tag]['total'] += stats_sup[tag]['total']

        print("Supervised model done")
    except Exception as e:
        print("Supervised failed:", e)

    print("\nSemi-supervised model is training")
    try:
        semi_tagger = train_semi_supervised(train_data, percent_tagged=1.0)
        stats_semi = evaluate_tagger(semi_tagger, test_data)

        for tag in stats_semi:
            results['semi_supervised'][tag]['correct'] += stats_semi[tag]['correct']
            results['semi_supervised'][tag]['total'] += stats_semi[tag]['total']

        print("Semi-supervised model done")
    except Exception as e:
        print("Semi-supervised failed:", e)

    fold_num += 1





----------------------------------------
Running fold 1
----------------------------------------
Train size: 4000
Test size: 1000

Unsupervised model is training
iteration 0 logprob -1188797.209474656
iteration 1 logprob -879011.6525562902
iteration 2 logprob -877965.4529938978
iteration 3 logprob -876863.1630676094
iteration 4 logprob -875515.8243630201
Unsupervised model done

Supervised model is training


/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


Supervised model done

Semi-supervised model is training
iteration 0 logprob -4.484600000000029e+304
iteration 1 logprob -6.391436595342868e+289
iteration 2 logprob -892037.196691362
iteration 3 logprob -865828.7819085391
iteration 4 logprob -863749.4419701553


/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)


Semi-supervised model done

----------------------------------------
Running fold 2
----------------------------------------
Train size: 4000
Test size: 1000

Unsupervised model is training
iteration 0 logprob -1189919.9866047269
iteration 1 logprob -876590.453794626
iteration 2 logprob -875162.0562426677
iteration 3 logprob -873712.170030436
iteration 4 logprob -871983.1681384519
Unsupervised model done

Supervised model is training
Supervised model done

Semi-supervised model is training
iteration 0 logprob -4.47010000000003e+304
iteration 1 logprob -6.2580809189905805e+289
iteration 2 logprob -892858.1622283141
iteration 3 logprob -863630.3271532261
iteration 4 logprob -861515.4034895102
Semi-supervised model done

----------------------------------------
Running fold 3
----------------------------------------
Train size: 4000
Test size: 1000

Unsupervised model is training
iteration 0 logprob -1199822.651000436
iteration 1 logprob -885799.1802255481
iteration 2 logprob -884784.1363

**Question 1.1:** In an unsupervised HMM with 12 hidden states, you replace the intended PoS tags with meaningless labels like S1–S12. How would this change affect the model’s learned behaviour and its evaluation accuracy? (4 points, 3-4 sentences)

The model would still learn patterns because an unsupervised HMM does not care about state names, but labels like S1-S12 would not match real PoS tags, so evaluation accuracy would be low unless the states are mapped to the true tags.

**Question 1.2:** You initialize a semi-supervised HMM with a fully supervised model and then run Baum–Welch only on unlabeled data. How can this additional unsupervised training change the model’s behaviour and accuracy? (4 points, 3-4 sentences)

Unsupervised training lets the model learn more patterns from the unlabeled data. This can help it work better than using only a small labeled set. Accuracy may increase, but in some cases it can also get worse.

**Question 1.3:** Consider the following four changes to an unsupervised or semi-supervised HMM for PoS tagging. For each one, state whether it would make training faster, slower, or have no significant effect, and justify your choice in one sentence. (4 points, 4 sentences)
- Reducing the number of hidden states from 12 to 6.
- Initializing the model with a fully supervised HMM instead of random parameters.
- Increasing the size of the unlabeled corpus by a factor of 5.
- Replacing full EM with a fixed small number of EM iterations (e.g., exactly 3 passes).

1)Reducing the number of hidden states from 12 to 6 would make training faster because there are fewer states to update during EM.

2)Initializing the model with a fully supervised HMM would make training faster because the model starts with better parameters.

3) Increasing the size of the unlabeled corpus by a factor of 5 would make training slower because EM has to process much more data.

4)Replacing full EM with a fixed small number of iterations would make training faster because the algorithm stops early instead of fully converging.

## 1.6: Display Results (5 points)

Display the average across folds results in a formatted table showing
- tag frequency (%)
- per-tag accuracy (%)
- overall accuracy (%)


1.   List item
2.   List item


for all three models.

In [58]:
def display_results(results):
    print("\nTAG    FREQ(%) |  UNSUP(%)   SUP(%)   SEMI(%)")
    print("-" * 55)

    total_tokens = 0
    for tag in results['supervised']:
        total_tokens += results['supervised'][tag]['total']

    for tag in sorted(results['supervised']):
        freq = 0
        if total_tokens > 0:
            freq = (results['supervised'][tag]['total'] / total_tokens) * 100

        if results['unsupervised'][tag]['total'] > 0:
            unsup = (results['unsupervised'][tag]['correct'] /
                     results['unsupervised'][tag]['total']) * 100
        else:
            unsup = 0

        if results['supervised'][tag]['total'] > 0:
            sup = (results['supervised'][tag]['correct'] /
                   results['supervised'][tag]['total']) * 100
        else:
            sup = 0

        if results['semi_supervised'][tag]['total'] > 0:
            semi = (results['semi_supervised'][tag]['correct'] /
                    results['semi_supervised'][tag]['total']) * 100
        else:
            semi = 0

        print(f"{tag:<5} {freq:7.2f} | {unsup:9.2f} {sup:8.2f} {semi:8.2f}")

    print("-" * 55)

    for model in results:
        total_correct = 0
        total_total = 0

        for tag in results[model]:
            total_correct += results[model][tag]['correct']
            total_total += results[model][tag]['total']

        if total_total > 0:
            acc = (total_correct / total_total) * 100
        else:
            acc = 0

        print(model, "overall accuracy:", f"{acc:.2f}%")

display_results(results)



TAG    FREQ(%) |  UNSUP(%)   SUP(%)   SEMI(%)
-------------------------------------------------------
.       11.68 |      1.93    33.45     3.09
ADJ      6.82 |      3.63    35.69     8.39
ADP     12.32 |      9.99    40.17    17.17
ADV      3.42 |      4.27    43.29     7.23
CONJ     2.73 |      3.20    35.10    15.54
DET     11.42 |     53.29    99.55    63.65
NOUN    30.17 |      2.98    36.19     4.86
NUM      2.06 |      5.85    39.49     7.05
PRON     2.56 |      6.03    56.18    32.61
PRT      2.25 |      1.76    40.96    25.28
VERB    14.47 |      2.75    44.75    17.43
X        0.09 |      9.09    23.23     0.00
-------------------------------------------------------
unsupervised overall accuracy: 9.64%
supervised overall accuracy: 45.69%
semi_supervised overall accuracy: 16.53%


## 1.7: Sample Predictions

Test the models on sample sentences to see how it performs.

In [51]:
tagger_sup = sup_tagger
tagger_unsup = unsup_tagger
tagger_semi_sup = semi_tagger

# Print sample predictions from the last fold
print("Sample predictions from last fold:")
sample_sentences = [
    "Today is a good day .",
    "Joe met Joanne in Delhi .",
    "Time flies like an arrow .",
    "The good , the bad , the ugly went to a bar ."
]

print(f"{'='*10} Spervised HMM Tagger {'='*10}")
for sent in sample_sentences:
    try:
        tagged = tagger_sup.tag(sent.split())
        print(f"  {sent}")
        print(f"  → {tagged}\n")
    except Exception as e:
        print(f"  {sent}")
        print(f"ERROR: Tagging failed: {e}\n")

print(f"{'='*10} Unspervised HMM Tagger {'='*10}")
for sent in sample_sentences:
    try:
        tagged = tagger_unsup.tag(sent.split())
        print(f"  {sent}")
        print(f"  → {tagged}\n")
    except Exception as e:
        print(f"  {sent}")
        print(f"ERROR: Tagging failed: {e}\n")

print(f"{'='*10} Semi-spervised HMM Tagger {'='*10}")
for sent in sample_sentences:
    try:
        tagged = tagger_semi_sup.tag(sent.split())
        print(f"  {sent}")
        print(f"  → {tagged}\n")
    except Exception as e:
        print(f"  {sent}")
        print(f"ERROR: Tagging failed: {e}\n")

Sample predictions from last fold:
========== Spervised HMM Tagger ==========
  Today is a good day .
  → [('Today', 'NOUN'), ('is', 'VERB'), ('a', 'DET'), ('good', 'ADJ'), ('day', 'NOUN'), ('.', '.')]

  Joe met Joanne in Delhi .
  → [('Joe', 'NOUN'), ('met', 'VERB'), ('Joanne', 'NOUN'), ('in', 'ADP'), ('Delhi', 'NOUN'), ('.', '.')]

  Time flies like an arrow .
  → [('Time', 'NOUN'), ('flies', 'VERB'), ('like', 'ADP'), ('an', 'DET'), ('arrow', 'DET'), ('.', 'DET')]

  The good , the bad , the ugly went to a bar .
  → [('The', 'DET'), ('good', 'ADJ'), (',', '.'), ('the', 'DET'), ('bad', 'ADJ'), (',', '.'), ('the', 'DET'), ('ugly', 'ADJ'), ('went', 'VERB'), ('to', 'ADP'), ('a', 'DET'), ('bar', 'NOUN'), ('.', '.')]

========== Unspervised HMM Tagger ==========
  Today is a good day .
  → [('Today', 'NOUN'), ('is', 'ADP'), ('a', 'NUM'), ('good', 'VERB'), ('day', 'CONJ'), ('.', 'X')]

  Joe met Joanne in Delhi .
  → [('Joe', 'NOUN'), ('met', 'X'), ('Joanne', 'NOUN'), ('in', 'ADP'), ('Delh

---

# Task 2: Text Generation with HMMs (24 points)

In this part, you will use a trained HMM to generate text. The idea is to sample from the learned transition and emission probabilities to create new sequences.

## 2.1: Train HMM Model for Generation

First, train a supervised HMM model on the Brown corpus for text generation. We'll normalise words to lowercase for better generation.

In [52]:
def train_hmm_model(num_sentences=5000):
    """Train an HMM model on Brown corpus for text generation"""
    print("Loading Brown corpus...")
    tagged_sents = list(brown.tagged_sents(tagset="universal"))[:num_sentences]

    print(f"Training HMM on {len(tagged_sents)} sentences...")

    # Extract states and symbols
    all_tags = set()
    all_symbols = set()
    for sent in tagged_sents:
        for word, tag in sent:
            all_tags.add(tag)
            all_symbols.add(word.lower())  # Normalise to lowercase

    # Normalise training data to lowercase
    normalized_sents = []
    for sent in tagged_sents:
        normalized_sents.append([(word.lower(), tag) for word, tag in sent])

    # Train the model
    trainer = hmm.HiddenMarkovModelTrainer(
        states=list(all_tags),
        symbols=list(all_symbols)
    )

    tagger = trainer.train_supervised(normalized_sents)

    print("Training complete!")
    return tagger

# Train the model
tagger_gen = train_hmm_model(num_sentences=20000)

# Show model statistics
print(f"\nModel Statistics:")
print(f"  - Number of states (POS tags): {len(tagger_gen._states)}")
print(f"  - Number of symbols (words): {len(tagger_gen._symbols)}")
print(f"  - States: {', '.join(sorted(tagger_gen._states))}")

Loading Brown corpus...
Training HMM on 20000 sentences...
Training complete!

Model Statistics:
  - Number of states (POS tags): 12
  - Number of symbols (words): 30743
  - States: ., ADJ, ADP, ADV, CONJ, DET, NOUN, NUM, PRON, PRT, VERB, X


## 2.2: Implement State Sampling (7 points)

Implement the `sample_state()` function that samples the next POS tag based on transition probabilities.

**Temperature parameter:** Controls randomness
- Lower temperature (e.g., 0.5): More conservative, follows high-probability transitions
- Higher temperature (e.g., 2.0): More creative, explores diverse transitions

In [53]:
def sample_state(tagger, current_state, temperature=1.0):
    """
    Sample next state from transition distribution.

    Args:
        tagger: Trained HMM tagger
        current_state: Current POS tag
        temperature: Controls randomness (default 1.0)

    Returns:
        Next POS tag (state)
    """

    states = list(tagger._states)


    log_probs = []
    for s in states:
        log_p = tagger._transitions[current_state].logprob(s)
        log_probs.append(log_p)

    log_probs = np.array(log_probs)

    log_probs = log_probs / temperature


    probs = 2 ** log_probs


    probs = probs / probs.sum()


    next_state = np.random.choice(states, p=probs)

    return next_state

## 2.3: Implement Word Sampling (7 points)

Implement the `sample_word()` function that samples a word given a POS tag based on emission probabilities.

In [54]:
def sample_word(tagger, state, temperature=1.0):
    """
    Sample word from output distribution of given state.

    Args:
        tagger: Trained HMM tagger
        state: Current POS tag
        temperature: Controls randomness (default 1.0)

    Returns:
        Sampled word
    """

    symbols = list(tagger._symbols)

    log_probs = []
    for sym in symbols:
        log_p = tagger._outputs[state].logprob(sym)
        log_probs.append(log_p)

    log_probs = np.array(log_probs)


    log_probs = log_probs / temperature


    probs = 2 ** log_probs

    probs = np.nan_to_num(probs)


    if probs.sum() == 0:
        probs = np.ones(len(probs)) / len(probs)
    else:
        probs = probs / probs.sum()


    word = np.random.choice(symbols, p=probs)

    return word

## 2.4: Implement Text Generation (10 points)

Implement the `generate_text_from_word()` function that generates text starting from a given word.

In [55]:
def generate_text_from_word(tagger, start_word, length=20, temperature=1.0):
    """
    Generate text starting with a given word using the HMM model.

    Args:
        tagger: Trained HMM tagger
        start_word: Word to start the generation
        length: Number of words to generate
        temperature: Controls randomness (higher = more random)

    Returns:
        List of generated words
    """
    start_word = start_word.lower()

    words = list(tagger._symbols)
    tags = list(tagger._states)

    if start_word not in words:
        start_word = np.random.choice(words)

    result = []
    result.append(start_word)

    current_tag = None
    max_prob = -1e100

    for tag in tags:
        p = tagger._outputs[tag].logprob(start_word)
        if p > max_prob:
            max_prob = p
            current_tag = tag

    for i in range(length - 1):
        next_tag = sample_state(tagger, current_tag, temperature)
        next_word = sample_word(tagger, next_tag, temperature)

        result.append(next_word)
        current_tag = next_tag

    return result

## 2.6: Test Text Generation

Now test your text generation functions with different starting words and temperatures.

In [56]:
# Test with example words
print("="*70)
print("EXAMPLE GENERATIONS")
print("="*70)

example_words = ["the", "dog", "running", "beautiful", "computer", "yesterday"]

for word in example_words:
    print(f"\n--- Starting with '{word}' ---")
    try:
        words = generate_text_from_word(tagger_gen, word, length=15, temperature=1.0)
        print(" ".join(words))
    except Exception as e:
        print(f"Error: {e}")

EXAMPLE GENERATIONS

--- Starting with 'the' ---
the churches . figures chili of workshop , found 13 destroy in contemplating the barrel

--- Starting with 'dog' ---
dog despite a bagley -- 18 rather is playhouse are the approach . he united

--- Starting with 'running' ---
running universities macwhorter in warm board kind yesterday , help . and justify off attend

--- Starting with 'beautiful' ---
beautiful communion . . world . and the rate b. is , the stripes graduates

--- Starting with 'computer' ---
computer -- again to for human math says down aid in wavelengths , of her

--- Starting with 'yesterday' ---
yesterday , if was for glazer , marmara it answered fanned the reach fringe designed


In [57]:
# Test different temperatures
print("\n" + "="*70)
print("TEMPERATURE COMPARISON")
print("="*70)

test_word = "the"
temperatures = [("Conservative", 0.5), ("Balanced", 1.0), ("Creative", 2.0)]

for temp_name, temp_value in temperatures:
    print(f"\n{temp_name} (temp={temp_value}):")
    words = generate_text_from_word(tagger_gen, test_word, length=20, temperature=temp_value)
    print(" ".join(words))


TEMPERATURE COMPARISON

Conservative (temp=0.5):
the men , the state , , was to is of the head . is of the children imagination ,

Balanced (temp=1.0):
the equity days of this bitter weights but good appreciation feeding they off other murtaugh remedy was in `` ,

Creative (temp=2.0):
the sibling nearer that selecting , outclassed him to decisive chain '' over done on or the between graduate ,


**Question 2.1:** How does the temperature parameter affect the quality and creativity of generated text? Provide two specific examples from your outputs. (4 points, 5-6 sentences)

The temperature controls how random the generated text is. When the temperature is low, the model mostly picks common words, so the text looks more correct but gets repetitive.

# Convert Your Colab Notebook to PDF

### Step 1: Download Your Notebook
- Go to **File → Download → Download .ipynb**
- Save the file to your computer

### Step 2: Upload to Colab
- Click the **📁 folder icon** on the left sidebar
- Click the **upload button**
- Select your downloaded .ipynb file
- Wait for the upload to complete

### Step 3: Run the Code Below
- **Uncomment the cell below** and run the cell
- This will take about 1-2 minutes to install required packages

### Step 4: Enter Notebook Name
- When prompted, type your notebook name (e.g.`gs_000000_as2.ipynb`)
- Press Enter

### The PDF will automatically download to your computer

In [ ]:
# # Install required packages (this takes about 30 seconds)
# print("Installing PDF converter... please wait...")
# !apt-get update -qq
# !apt-get install -y texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc > /dev/null 2>&1
# !pip install -q nbconvert

# print("\n" + "="*50)
# print("COLAB NOTEBOOK TO PDF CONVERTER")
# print("="*50)
# print("\nSTEP 1: Download your notebook")
# print("- Go to File → Download → Download .ipynb")
# print("- Save it to your computer")
# print("\nSTEP 2: Upload it here")
# print("- Click the folder icon on the left (📁)")
# print("- Click the upload button and select your .ipynb file")
# print("- Wait for upload to complete")
# print("\nSTEP 3: Enter the filename below")
# print("="*50)

# # Get notebook name from user
# notebook_name = input("\nEnter your notebook name: ")

# # Add .ipynb if missing
# if not notebook_name.endswith('.ipynb'):
#     notebook_name += '.ipynb'

# import os
# notebook_path = f'/content/{notebook_name}'

# # Check if file exists
# if not os.path.exists(notebook_path):
#     print(f"\n⚠ Error: '{notebook_name}' not found in /content/")
#     print("\nMake sure you uploaded the file using the folder icon (📁) on the left!")
# else:
#     print(f"\n✓ Found {notebook_name}")
#     print("Converting to PDF... this may take 1-2 minutes...\n")

#     # Convert the notebook to PDF
#     !jupyter nbconvert --to pdf "{notebook_path}"

#     # Download the PDF
#     from google.colab import files
#     pdf_name = notebook_name.replace('.ipynb', '.pdf')
#     pdf_path = f'/content/{pdf_name}'

#     if os.path.exists(pdf_path):
#         print("✓ SUCCESS! Downloading your PDF now...")
#         files.download(pdf_path)
#         print("\n✓ Done! Check your downloads folder.")
#     else:
#         print("⚠ Error: Could not create PDF")